# Agentic RAG System Demo

This notebook demonstrates a 2-agent system for Retrieval-Augmented Generation (RAG):

1. **RetrieverAgent**: Retrieves chunks from Pinecone (namespace: 'no-excel')
2. **EvaluatorAgent**: Evaluates sufficiency and generates answers using Gemini
3. **AgenticRAGOrchestrator**: Coordinates iterative retrieval-evaluation loops

The system processes sample questions and compares results with expected answers.

## 1. Setup and Imports

In [ ]:
import os
from pathlib import Path
from datetime import datetime
import json

# Data handling
import pandas as pd
from dotenv import load_dotenv

# AI & Vector DB
import torch
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
import google.generativeai as genai

# Our custom agentic RAG system
from agentic_rag import RetrieverAgent, EvaluatorAgent, AgenticRAGOrchestrator

print("All imports successful!")

## 2. Load Environment Variables

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Load API keys
try:
    PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
    PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME')
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

    if not all([PINECONE_API_KEY, PINECONE_INDEX_NAME, GEMINI_API_KEY]):
        raise ValueError("Missing required environment variables")

    # Configure Gemini
    genai.configure(api_key=GEMINI_API_KEY)
    print("✓ Environment variables loaded successfully")

except Exception as e:
    print(f"✗ Error loading environment variables: {e}")

## 3. Initialize Pinecone and Embedding Model

In [ ]:
# Configuration
MODEL_NAME = "all-mpnet-base-v2"
NAMESPACE = "no-excel"  # Using the no-excel namespace
NAMESPACE2 = ''
# Initialize Pinecone
try:
    print("Initializing Pinecone client...")
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index = pc.Index(PINECONE_INDEX_NAME)
    
    print(f"\nConnected to index: {PINECONE_INDEX_NAME}")
    print("Index stats:")
    stats = index.describe_index_stats()
    print(json.dumps(stats, indent=2))
    
    # Check if no-excel namespace exists
    if NAMESPACE in stats.get('namespaces', {}):
        namespace_count = stats['namespaces'][NAMESPACE]['vector_count']
        print(f"\n✓ Namespace '{NAMESPACE}' found with {namespace_count} vectors")
    else:
        print(f"\n⚠ Warning: Namespace '{NAMESPACE}' not found in index")
        print(f"Available namespaces: {list(stats.get('namespaces', {}).keys())}")
    
except Exception as e:
    print(f"✗ Error initializing Pinecone: {e}")

# Load Embedding Model
try:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nUsing device: {device}")
    print(f"Loading embedding model: {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME, device=device)
    print("✓ Embedding model loaded successfully")
except Exception as e:
    print(f"✗ Error loading embedding model: {e}")

## 4. Initialize Gemini Model

In [ ]:
try:
    generation_config = {
        "temperature": 0.2,
        "top_p": 1,
        "top_k": 5,
        "max_output_tokens": 2048,
    }
    
    gemini_model = genai.GenerativeModel(
        model_name="gemini-2.0-flash-lite",
        generation_config=generation_config
    )
    print("✓ Gemini model initialized successfully")
except Exception as e:
    print(f"✗ Error initializing Gemini model: {e}")

## 5. Initialize Agents and Orchestrator

In [ ]:
# Create agents
retriever = RetrieverAgent(
    index=index,
    embedding_model=model,
    namespace=NAMESPACE
)

evaluator = EvaluatorAgent(
    gemini_model=gemini_model
)

# Create orchestrator
orchestrator = AgenticRAGOrchestrator(
    retriever=retriever,
    evaluator=evaluator,
    max_iterations=3,
    initial_top_k=1,
    top_k_increment=3
)

print("✓ All agents initialized successfully")
print(f"  - RetrieverAgent (namespace: {NAMESPACE})")
print(f"  - EvaluatorAgent (model: gemini-2.0-flash-exp)")
print(f"  - Orchestrator (max_iterations: 3, initial_top_k: 5)")

## 6. Define Sample Questions and Expected Answers

In [ ]:
sample_questions = [
    "What is the Corporate Identity Number (CIN) of the company?",
    "What is the authorised capital of the company in Rupees as of the filing date?",
    "When was the Annual General Meeting (AGM) held?",
    "Where is the registered office of the company located?",
    "Whether Secretarial Audit is applicable",
    "Which financial year do these financial statements cover?",
    "What is the professional status of the certifier (e.g., Chartered Accountant, Company Secretary)?",
]

expected_answers = [
    "The Corporate Identity Number (CIN) is U00000XX0000PTC000000.",
    "The authorized capital of the company is 5,000,000.0 Rs.",
    "The Annual General Meeting (AGM) was held on 30/09/2024.",
    "The registered office address is PLOT NO.1, EXAMPLE INDUSTRIAL ESTATE, EXAMPLE, EXAMPLE CITY, Karnataka, 560000, India.",
    "Yes, a Secretarial Audit is applicable.",
    "The financial year covered is from 01/04/2023 to 31/03/2024.",
    "The certifying professional is a Chartered accountant (in whole-time practice) with the status of a Fellow."
]

print(f"Prepared {len(sample_questions)} questions for evaluation")

## 7. Process Questions with Agentic RAG System

In [ ]:
# Process all questions
results = orchestrator.process_questions_batch(sample_questions)

print("\n" + "="*80)
print("PROCESSING COMPLETE")
print("="*80)

## 8. Evaluate Results

In [ ]:
# Create evaluation dataframe
evaluation_data = []

for idx, result in enumerate(results):
    evaluation_data.append({
        'Question_Number': idx + 1,
        'Question': result['question'],
        'Expected_Answer': expected_answers[idx],
        'Generated_Answer': result['answer'],
        'Iterations': result['iterations'],
        'Total_Chunks_Used': result['total_chunks_used'],
        'Success': result['success'],
        'Note': result.get('note', '')
    })

df_evaluation = pd.DataFrame(evaluation_data)

# Display results
print("\nEvaluation Summary:")
print(f"Total Questions: {len(results)}")
print(f"Successful: {df_evaluation['Success'].sum()}")
print(f"Average Iterations: {df_evaluation['Iterations'].mean():.2f}")
print(f"Average Chunks Used: {df_evaluation['Total_Chunks_Used'].mean():.1f}")

# Display dataframe
print("\nDetailed Results:")
df_evaluation

## 9. Display Side-by-Side Comparison

In [ ]:
# Create comparison view
print("\n" + "="*80)
print("SIDE-BY-SIDE COMPARISON")
print("="*80 + "\n")

for idx, row in df_evaluation.iterrows():
    print(f"Question {row['Question_Number']}: {row['Question']}")
    print("-" * 80)
    print(f"Expected:\n{row['Expected_Answer']}")
    print(f"\nGenerated:\n{row['Generated_Answer']}")
    print(f"\nMetrics: Iterations={row['Iterations']}, Chunks={row['Total_Chunks_Used']}, Success={row['Success']}")
    print("\n" + "="*80 + "\n")

## 10. Save Results to CSV

In [ ]:
# Create results directory if it doesn't exist
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Generate timestamp for filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = results_dir / f"agentic_rag_results_{timestamp}.csv"

# Save main results
df_evaluation.to_csv(csv_filename, index=False)
print(f"✓ Results saved to: {csv_filename}")

# Save detailed logs
logs_filename = results_dir / f"agentic_rag_logs_{timestamp}.json"
with open(logs_filename, 'w') as f:
    json.dump(results, f, indent=2)
print(f"✓ Detailed logs saved to: {logs_filename}")

## 11. Visualize Iteration Distribution

In [ ]:
import matplotlib.pyplot as plt

# Create visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Iteration distribution
iteration_counts = df_evaluation['Iterations'].value_counts().sort_index()
axes[0].bar(iteration_counts.index, iteration_counts.values, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Number of Iterations')
axes[0].set_ylabel('Number of Questions')
axes[0].set_title('Distribution of Iterations')
axes[0].grid(axis='y', alpha=0.3)

# Success rate
success_counts = df_evaluation['Success'].value_counts()
axes[1].pie(success_counts.values, labels=['Successful' if x else 'Partial' for x in success_counts.index], 
            autopct='%1.1f%%', colors=['green', 'orange'], startangle=90)
axes[1].set_title('Success Rate')

plt.tight_layout()
plt.savefig(results_dir / f"agentic_rag_visualization_{timestamp}.png", dpi=150, bbox_inches='tight')
print(f"✓ Visualization saved to: {results_dir / f'agentic_rag_visualization_{timestamp}.png'}")
plt.show()

## 12. Test Individual Question (Optional)

In [ ]:
# Test a single question with detailed output
test_question = "What is the Corporate Identity Number (CIN) of the company?"

print(f"Testing single question: {test_question}\n")
result = orchestrator.process_question(test_question)

print("\n" + "="*80)
print("RESULT")
print("="*80)
print(f"Answer: {result['answer']}")
print(f"Iterations: {result['iterations']}")
print(f"Success: {result['success']}")
print(f"\nDetailed Logs:")
print(json.dumps(result['logs'], indent=2))